In [ ]:
import os
import time
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt




#API_KEY = "a8c86112313046389464f4b8ffa4ff7c"
#INPUT = "top10pct_priority_ice_slush_weighted.csv"

#MAX_REQUESTS = 3000  # Geoapify free 3000 requests per day

GEOCODED_OUT = "/Users/yiyuanemmazhou/Documents/New Project/top10pct_geocoded_geoapify.csv"
AREA_COUNTS_OUT = "/Users/yiyuanemmazhou/Documents/New Project/top10pct_area_counts_ranked.csv"
MAP_OUT = "/Users/yiyuanemmazhou/Documents/New Project/top10pct_map.png"

# Boston neighborhoods GeoJSON
NEIGHBORHOODS_URL = (
    "https://gis.bostonplans.org/hosting/rest/services/Hosted/"
    "Data_2025_Neighborhood_AnalyzeB/FeatureServer/0/query"
    "?where=1%3D1&outFields=name&outSR=4326&f=geojson"
)





df = pd.read_csv(INPUT)
inter_col = [c for c in df.columns if "intersection" in c.lower()][0]
intersections = df[[inter_col]].drop_duplicates().rename(columns={inter_col: "intersection"})

if os.path.exists(GEOCODED_OUT):
    cache = pd.read_csv(GEOCODED_OUT)
else:
    cache = pd.DataFrame(columns=["intersection", "lat", "lon"])


def geocode_one(query):
    url = "https://api.geoapify.com/v1/geocode/search"
    params = {"text": query, "apiKey": API_KEY, "format": "json", "limit": 1}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    if "results" in data and len(data["results"]) > 0:
        return data["results"][0]["lat"], data["results"][0]["lon"]
    return None, None


# geocode missing ones
to_do = intersections[~intersections["intersection"].isin(cache["intersection"])]
if MAX_REQUESTS:
    to_do = to_do.head(MAX_REQUESTS)

new_rows = []
for _, row in to_do.iterrows():
    q = f"{row['intersection']}, Boston, MA"
    lat, lon = geocode_one(q)
    new_rows.append({"intersection": row["intersection"], "lat": lat, "lon": lon})
    time.sleep(0.25)

if new_rows:
    cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)
    cache.to_csv(GEOCODED_OUT, index=False)


geo = cache.dropna(subset=["lat", "lon"]).copy()
gdf_pts = gpd.GeoDataFrame(
    geo,
    geometry=[Point(xy) for xy in zip(geo["lon"], geo["lat"])],
    crs="EPSG:4326"
)


gdf_nbhd = gpd.read_file(NEIGHBORHOODS_URL)


joined = gpd.sjoin(gdf_pts, gdf_nbhd[["name", "geometry"]], how="left", predicate="within")


area_counts = (
    joined.groupby("name")
    .size()
    .rename("intersection_count")
    .sort_values(ascending=False)
    .reset_index()
)
area_counts["rank"] = range(1, len(area_counts) + 1)

area_counts.to_csv(AREA_COUNTS_OUT, index=False)

print("Top areas by # of high‑risk intersections:")
print(area_counts.head(10))




KeyboardInterrupt: 

In [ ]:

# code to make the .svg map (first look @ the map)

import json

geojson_path = "/Users/yiyuanemmazhou/Documents/New Project/boston_neighborhoods.geojson"
output_svg = "/Users/yiyuanemmazhou/Documents/New project/boston_selected_neighborhoods_map.svg"

selected = {
    "Dorchester",
    "Downtown",
    "East Boston",
    "Fenway",
    "South Boston Waterfront"
}

with open(geojson_path, "r") as f:
    gj = json.load(f)

features = gj.get("features", [])

polys = []

for feat in features:
    name = feat.get("properties", {}).get("name")
    geom = feat.get("geometry")
    if not geom:
        continue
    gtype = geom.get("type")
    coords = geom.get("coordinates")

    if gtype == "Polygon":
        outer = coords[0]
        polys.append((name, outer))
    elif gtype == "MultiPolygon":
        for poly in coords:
            outer = poly[0]
            polys.append((name, outer))


lons = [pt[0] for _, ring in polys for pt in ring]
lats = [pt[1] for _, ring in polys for pt in ring]
min_lon, max_lon = min(lons), max(lons)
min_lat, max_lat = min(lats), max(lats)

# SVG dimensions
width = 1000
height = 1000
padding = 20

sx = (width - 2 * padding) / (max_lon - min_lon)
sy = (height - 2 * padding) / (max_lat - min_lat)

def proj(lon, lat):
    x = padding + (lon - min_lon) * sx
    y = padding + (max_lat - lat) * sy
    return x, y





svg_parts = []
svg_parts.append(f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">')
svg_parts.append('<rect width="100%" height="100%" fill="white"/>')


for name, ring in polys:
    points = " ".join([f"{proj(lon,lat)[0]:.2f},{proj(lon,lat)[1]:.2f}" for lon,lat in ring])
    svg_parts.append(f'<polygon points="{points}" fill="#e8e8e8" stroke="#777777" stroke-width="0.6"/>')

# Selected neighborhoods
for name, ring in polys:
    if name not in selected:
        continue
    points = " ".join([f"{proj(lon,lat)[0]:.2f},{proj(lon,lat)[1]:.2f}" for lon,lat in ring])
    svg_parts.append(f'<polygon points="{points}" fill="#ff6b35" stroke="#333333" stroke-width="1.2"/>')

svg_parts.append('<text x="20" y="35" font-family="Arial" font-size="20" fill="#222">Selected Neighborhoods (>=21 Top-10% Intersections)</text>')
svg_parts.append("</svg>")

with open(output_svg, "w") as f:
    f.write("\n".join(svg_parts))

print("saved", output_svg)


In [ ]:
from google.colab import files
files.upload()


Saving top10pct_priority_ice_slush_weighted.csv to top10pct_priority_ice_slush_weighted.csv


{'top10pct_priority_ice_slush_weighted.csv': b"intersection,N_i,ice_n,slush_n,ice_share,slush_share,priority_ice_slush,volume_weight,priority_ice_slush_weighted,rank,rank_top10pct\nWASHINGTON STREET,271,6.0,1.0,0.022140221402214,0.003690036900369,7.1671888980920855,5.605802066295998,40.17784233445835,1,1\nSOUTH STREET,108,3.0,1.0,0.0277777777777777,0.0092592592592592,4.089732194716624,4.6913478822291435,19.18635647056818,2,2\nADAMS STREET,104,3.0,1.0,0.0288461538461538,0.0096153846153846,4.089732194716625,4.653960350157523,19.033451476973877,3,3\nYANKEE DIVISION HIGHWAY Rte SR128,178,2.0,1.0,0.0112359550561797,0.0056179775280898,3.063913293591471,5.187385805840755,15.893700329503194,4,4\nCENTRE STREET,173,2.0,1.0,0.0115606936416184,0.0057803468208092,3.063913293591471,5.159055299214529,15.80689811363692,5,5\nNEWBURYPORT TURNPIKE Rte US1,39,3.0,1.0,0.0769230769230769,0.0256410256410256,4.089732194716625,3.688879454113936,15.086529065918452,6,6\nWALNUT STREET,126,2.0,1.0,0.01587301587301